In [2]:
print("cand cols:", cand.columns.tolist())
print("shap cols:", shap.columns.tolist())
print("deg  cols:", deg.columns.tolist())


cand cols: ['candidate_gene']
shap cols: ['gene', 'mean_abs_shap']
deg  cols: ['gene', 'pvalue', 'logFC']


In [4]:
BASE = "data/processed"
import os
import pandas as pd

# load artifacts
shap = pd.read_csv(f"{BASE}/shap_importance.csv")              # gene, mean_abs_shap
deg  = pd.read_csv(f"{BASE}/deg_label_based.csv")              # gene, pvalue, logFC
cand = pd.read_csv(f"{BASE}/candidate_genes_intersection.csv") # candidate_gene

# FIX: align column name for merging
cand = cand.rename(columns={"candidate_gene": "gene"})

# merge into final table
final = (
    cand.merge(shap, on="gene", how="left")
        .merge(deg,  on="gene", how="left")
        .sort_values("mean_abs_shap", ascending=False)
)

report_df = final.head(10)[["gene", "mean_abs_shap", "pvalue", "logFC"]].copy()
report_df.columns = ["Gene", "SHAP", "pvalue", "logFC"]

# write markdown
out_md = f"{BASE}/TCGA_BRCA_target_report.md"
with open(out_md, "w") as f:
    f.write("# TCGA BRCA Target Evidence Report\n\n")
    f.write(report_df.to_markdown(index=False))

out_md, report_df.head(10)


('data/processed/TCGA_BRCA_target_report.md',
      Gene      SHAP    pvalue     logFC
 18  G6993  0.351298  0.000303 -0.498101
 12  G4480  0.331959  0.000884 -0.469700
 6   G2951  0.289228  0.007352 -0.373229
 10   G400  0.230345  0.001608 -0.445602
 9   G3939  0.123591  0.008782 -0.374067
 11  G4457  0.118877  0.000490  0.488175
 4   G2048  0.108583  0.003316  0.417526
 15  G5925  0.105184  0.004004 -0.401351
 7   G3819  0.096648  0.000159  0.530542
 20  G7438  0.089751  0.003593  0.410555)

In [ ]:
# Select final candidates

final_genes = top_genes["gene"].head(10).tolist()
final_genes

In [ ]:
# Evidence placeholder (later: PubMed / RAG)

evidence = {
    g: f"Evidence placeholder for {g} (to be replaced by literature retrieval)"
    for g in final_genes
}
evidence

In [ ]:
# Build report table

import pandas as pd

report_df = pd.DataFrame({
    "Gene": final_genes,
    "Evidence": [evidence[g] for g in final_genes]
})

report_df

In [ ]:
# Write markdown report to disk

from src.report import write_markdown_report

sections = [("Candidate targets", report_df.to_markdown(index=False))]

write_markdown_report(
    "data/processed/TCGA_BRCA_target_report.md",
    "TCGA-BRCA Target Evidence Report",
    sections
)